# Building Energy Consumption Prediction
## Notebook 01: Exploratory Data Analysis

**Prerequisite:** Run `python src/data_collection.py` and `python src/preprocessing.py` first.

### Contents
1. Dataset overview and shapes
2. Meter reading distribution analysis
3. Building metadata statistics
4. Weather data quality and distributions
5. Missing values audit
6. Temporal patterns (hourly, daily, monthly)
7. Climate zone building counts
8. Feature correlation heatmap

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import DATA_RAW, DATA_CLEAN, SITE_CLIMATE_ZONE, METER_LABELS

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
sns.set_theme(style='whitegrid', palette='colorblind')
print('Libraries loaded.')

## 1. Dataset Overview

In [ ]:
ashrae_dir = DATA_RAW / 'ashrae'

train = pd.read_csv(ashrae_dir / 'train.csv', parse_dates=['timestamp'], nrows=500_000)
meta  = pd.read_csv(ashrae_dir / 'building_metadata.csv')
weather = pd.read_csv(ashrae_dir / 'weather_train.csv', parse_dates=['timestamp'])

print('=== ASHRAE GPED-III Dataset Overview ===')
print(f'\ntrain.csv      : {len(train):>10,} rows (500k sample) | {train.shape[1]} cols')
print(f'building_meta  : {len(meta):>10,} buildings         | {meta.shape[1]} cols')
print(f'weather_train  : {len(weather):>10,} rows            | {weather.shape[1]} cols')
print(f'\nMeter types: {train["meter"].value_counts().to_dict()}')
print(f'Sites (16): {sorted(meta["site_id"].unique())}')
print(f'Building use categories: {meta["primary_use"].nunique()}')
print(f'Date range: {train["timestamp"].min()} → {train["timestamp"].max()}')

In [ ]:
print('\n--- Building Metadata Sample ---')
display(meta.head(10))
print('\n--- Building Primary Use Distribution ---')
display(meta['primary_use'].value_counts())

## 2. Meter Reading Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution (log scale)
ax = axes[0]
for m, label in METER_LABELS.items():
    data = train[train['meter'] == m]['meter_reading']
    data = data[(data > 0) & (data < data.quantile(0.99))]
    ax.hist(data, bins=80, alpha=0.5, label=label, density=True)
ax.set_xlabel('Meter Reading (kWh)')
ax.set_ylabel('Density')
ax.set_title('Meter Reading Distribution by Type')
ax.set_xlim(0, None)
ax.legend()

# Log-transformed
ax = axes[1]
for m, label in METER_LABELS.items():
    data = train[train['meter'] == m]['meter_reading']
    data = np.log1p(data[(data >= 0)])
    ax.hist(data, bins=80, alpha=0.5, label=label, density=True)
ax.set_xlabel('log(1 + Meter Reading)')
ax.set_ylabel('Density')
ax.set_title('Log-Transformed Meter Reading Distribution')
ax.legend()

plt.tight_layout()
plt.suptitle('Figure EDA-1: Meter Reading Distributions (500k sample)', y=1.02, fontsize=12)
plt.show()
print('Log transformation normalises the highly right-skewed distributions → justifies log-transform of target.')

## 3. Building Metadata Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Floor area distribution
meta['square_feet'].plot(kind='hist', bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Gross Floor Area (sq ft)')
axes[0].set_title('Floor Area Distribution')

# Year built
meta['year_built'].dropna().plot(kind='hist', bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_xlabel('Year Built')
axes[1].set_title('Building Age Distribution')

# Primary use
use_counts = meta['primary_use'].value_counts().head(10)
use_counts.plot(kind='barh', ax=axes[2], color='teal', edgecolor='white')
axes[2].set_xlabel('Building Count')
axes[2].set_title('Top 10 Primary Use Types')

plt.tight_layout()
plt.suptitle('Figure EDA-2: Building Metadata Distributions', y=1.02, fontsize=12)
plt.show()

print(f'Floor area: min={meta["square_feet"].min():,.0f} | max={meta["square_feet"].max():,.0f} | median={meta["square_feet"].median():,.0f} sq ft')
print(f'Year built: {meta["year_built"].min():.0f} – {meta["year_built"].max():.0f} | missing={meta["year_built"].isna().sum()}')

## 4. Climate Zone Distribution

In [ ]:
meta['climate_zone'] = meta['site_id'].map(SITE_CLIMATE_ZONE)
zone_counts = meta['climate_zone'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('colorblind', len(zone_counts))
zone_counts.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Number of Buildings')
ax.set_title('Figure EDA-3: Building Count by ASHRAE Climate Zone')
for i, v in enumerate(zone_counts):
    ax.text(v + 5, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Climate zone distribution:')
print(zone_counts.to_string())

## 5. Missing Values Audit

In [ ]:
print('=== Missing Values Audit ===')
for name, df in [('train (500k)', train), ('building_metadata', meta), ('weather_train', weather)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f'\n{name}:')
    if len(missing) == 0:
        print('  No missing values.')
    else:
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f'  {col:35s}: {cnt:6,} ({pct:.1f}%)')

## 6. Temporal Patterns

In [ ]:
elec = train[train['meter'] == 0].copy()
elec['hour'] = elec['timestamp'].dt.hour
elec['month'] = elec['timestamp'].dt.month
elec['dayofweek'] = elec['timestamp'].dt.dayofweek

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

elec.groupby('hour')['meter_reading'].median().plot(ax=axes[0], marker='o', ms=4, color='steelblue')
axes[0].set_xlabel('Hour of Day'); axes[0].set_title('Median Electricity by Hour')

elec.groupby('month')['meter_reading'].median().plot(ax=axes[1], marker='o', ms=4, color='coral')
axes[1].set_xlabel('Month'); axes[1].set_title('Median Electricity by Month')

dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
elec.groupby('dayofweek')['meter_reading'].median().plot(
    ax=axes[2], kind='bar', color='teal', edgecolor='white')
axes[2].set_xticklabels(dow_labels, rotation=0)
axes[2].set_xlabel('Day of Week'); axes[2].set_title('Median Electricity by Day')

plt.tight_layout()
plt.suptitle('Figure EDA-4: Temporal Patterns in Electricity Consumption', y=1.02, fontsize=12)
plt.show()

## 7. Feature Correlation (Clean Dataset)

In [ ]:
clean_path = DATA_CLEAN / 'ashrae_clean.parquet'
if clean_path.exists():
    df_clean = pd.read_parquet(clean_path).sample(50_000, random_state=42)
    num_cols = ['meter_reading', 'air_temperature', 'log_square_feet', 'building_age',
                'HDD', 'CDD', 'temp_roll24h', 'wind_speed', 'dew_temperature',
                'hour', 'month', 'is_weekend', 'relative_humidity']
    num_cols = [c for c in num_cols if c in df_clean.columns]
    corr = df_clean[num_cols].corr()

    fig, ax = plt.subplots(figsize=(11, 9))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title('Figure EDA-5: Feature Correlation Matrix (50k sample)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Clean dataset not found. Run src/preprocessing.py first.')

## 8. Summary Statistics

In [ ]:
if clean_path.exists():
    print('=== Clean Dataset Summary ===')
    print(f'Shape: {df_clean.shape}')
    print(f'\nTarget (meter_reading) statistics:')
    print(df_clean['meter_reading'].describe().round(2).to_string())
    print(f'\nZero readings: {(df_clean["meter_reading"] == 0).sum():,} ({(df_clean["meter_reading"] == 0).mean()*100:.1f}%)')
    print(f'\nFeature columns available: {[c for c in df_clean.columns if c != "meter_reading"]}')